In [1]:
import pandas as pd
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.config import LEXICON_PATH, TRAIN_PARQUET_PATH
import re
import math
import tempfile
import os
from io import StringIO
from collections import Counter, defaultdict
from tqdm import tqdm
try:
    import eflomal
except ImportError:
    eflomal = None  # Existing lexicon remains usable on Windows.
import warnings
warnings.filterwarnings("ignore")

In [2]:
# =========================
# 1. TOKENIZATION
# =========================
def tokenize(text):
    return re.findall(r"\w+(?:[-'']\w+)*|[^\w\s]", str(text), flags=re.UNICODE)
 
def word_tokens(text):
    return re.findall(r"\w+(?:[-'']\w+)*", str(text).lower(), flags=re.UNICODE)
 
 
# =========================
# 2. WOLOF NORMALIZATION
# =========================
rules = [
    (r"(\w+)-a-(\w+)", r"\1a\2"),
    (r"-", r" "),
    (r"([aieo])\1", r"\1"),
    (r"uu", r"__OU__"),
    (r"ñ+", r"gn"),
    (r"ŋ+", r"ng"),
    (r"ë+", r"__EU__"),
    (r"ée", r"e"),
    (r"g([ieéèêë])", r"g__U__\1"),
    (r"ó", r"o"),
    (r"(à|á|â|ä)", r"a"),
    (r"u([blt]+)", r"__OU__\1"),
    (r"u+", r"__OU__"),
    (r"q", r"kh"),
    (r"x", r"kh"),
    (r"c([aeiouy]+)", r"th\1"),
    (r"c{2}\b", r"thie"),
    (r"[Jj]([eaoë]{1,2})", r"di\1"),
    (r"[Jj]([i]+)", r"dj\1"),
    (r"[Jj]([u]+)", r"dio\1"),
    (r"th([aeouy]+)", r"thi\1"),
    (r"e\b", r"é"),
    (r"n\b", r"ne"),
    (r"gg", r"g"),
    (r"__EU__", r"eu"),
    (r"__OU__", r"ou"),
    (r"__U__", r"u")
]
 
def normalize_wolof_token(token):
    out = token.lower()
    for pattern, repl in rules:
        out = re.sub(pattern, repl, out)
    return out

In [3]:
# =========================
# 3. ENTITY PROTECTION
# =========================
def mask_protected(text, entities=None):
    spans = []
    if entities:
        for ent in entities:
            spans.append((ent["start"], ent["end"]))
    for m in re.finditer(r"\b[A-ZÀ-Ý0-9]{2,}\b", text):
        spans.append((m.start(), m.end()))
    for m in re.finditer(r"\b\d+(?:[.,]\d+)?\b", text):
        spans.append((m.start(), m.end()))
    spans = sorted(spans)
    merged = []
    for s, e in spans:
        if not merged or s > merged[-1][1]:
            merged.append([s, e])
        else:
            merged[-1][1] = max(merged[-1][1], e)
    saved = {}
    masked = text
    for i, (s, e) in enumerate(reversed(merged)):
        token = f"__protected_{i}__"
        saved[token] = masked[s:e]
        masked = masked[:s] + token + masked[e:]
    return masked, saved
 
def restore_protected(text, saved):
    for token, value in saved.items():
        text = text.replace(token, value)
    return text
 
 
# =========================
# 4. EFLOMAL ALIGNMENT
# =========================
def run_eflomal(src_lines, trg_lines):
    """
    Run eflomal on a list of source/target sentence pairs.
    Returns forward alignments as a list of dicts: {src_idx: trg_idx}
    """
    src_text = "\n".join(src_lines) + "\n"
    trg_text = "\n".join(trg_lines) + "\n"
 
    with tempfile.NamedTemporaryFile(mode='w', suffix='.fwd', delete=False) as fwd_f, \
         tempfile.NamedTemporaryFile(mode='w', suffix='.rev', delete=False) as rev_f:
        fwd_name = fwd_f.name
        rev_name = rev_f.name
 
    try:
        aligner = eflomal.Aligner()
        aligner.align(
            StringIO(src_text),
            StringIO(trg_text),
            links_filename_fwd=fwd_name,
            links_filename_rev=rev_name,
            quiet=True
        )
        alignments = []
        with open(fwd_name) as f:
            for line in f:
                pairs = {}
                for link in line.strip().split():
                    if '-' in link:
                        s, t = link.split('-')
                        pairs[int(s)] = int(t)
                alignments.append(pairs)
        return alignments
    finally:
        os.unlink(fwd_name)
        os.unlink(rev_name)

In [4]:
# =========================
# 5. BUILD LEXICON FROM ALIGNMENTS
# =========================
import pickle
from pathlib import Path

def build_lexicon(df, wolof_col="wolof", french_col="french",
                  min_count=3, min_prob=0.1, batch_size=5000, save_path=LEXICON_PATH):
    """
    Run eflomal over the full corpus in batches and build a
    Wolof -> {french_word: probability} lexicon.
    """
    # co-occurrence counts: wolof_word -> Counter(french_word)
    cooc = defaultdict(Counter)
    wo_total = Counter()  # how many times each Wolof word was aligned to anything
 
    src_lines = [str(r).lower().strip() for r in df[wolof_col]]
    trg_lines = [str(r).lower().strip() for r in df[french_col]]
 
    n = len(src_lines)
    print(f"Running eflomal alignment on {n} sentence pairs...")
 
    for start in tqdm(range(0, n, batch_size)):
        end = min(start + batch_size, n)
        batch_src = src_lines[start:end]
        batch_trg = trg_lines[start:end]
 
        alignments = run_eflomal(batch_src, batch_trg)
 
        for i, pairs in enumerate(alignments):
            src_toks = word_tokens(batch_src[i])
            trg_toks = word_tokens(batch_trg[i])
            for s_idx, t_idx in pairs.items():
                if s_idx < len(src_toks) and t_idx < len(trg_toks):
                    wo_word = src_toks[s_idx]
                    fr_word = trg_toks[t_idx]
                    cooc[wo_word][fr_word] += 1
                    wo_total[wo_word] += 1
 
    # Normalize to probabilities, filter by min_count and min_prob
    lexicon = {}
    for wo_word, fr_counts in cooc.items():
        total = wo_total[wo_word]
        filtered = {
            fr: count / total
            for fr, count in fr_counts.items()
            if count >= min_count and count / total >= min_prob
        }
        if filtered:
            lexicon[wo_word] = filtered

    print(f"Lexicon built: {len(lexicon)} Wolof words with French translations")

    # Save to disk
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    with open(save_path, "wb") as f:
        pickle.dump(lexicon, f)

    print(f"Lexicon saved to: {save_path}")

    return lexicon
 
 
# =========================
# 6. CANDIDATE SELECTION
# =========================
def get_substitution_candidates(wolof_tokens, french_tokens, lexicon,
                                 num_substitutions=2):
    """
    For each Wolof token, find its best French translation that:
    1. Exists in the lexicon
    2. Also appears in the paired French sentence (constraint from original design)
    3. Has not already been used as a substitution target
 
    Returns a list of (wolof_idx, french_word, probability) sorted by probability desc.
    """
    french_set = set(french_tokens)
    candidates = []
 
    for i, wo_tok in enumerate(wolof_tokens):
        if wo_tok not in lexicon:
            continue
 
        translations = lexicon[wo_tok]  # {fr_word: prob}
 
        # Only keep translations that appear in the paired French sentence
        constrained = {
            fr: prob for fr, prob in translations.items()
            if fr in french_set
        }
 
        if not constrained:
            continue
 
        best_fr = max(constrained, key=constrained.get)
        best_prob = constrained[best_fr]
 
        candidates.append((i, best_fr, best_prob))
 
    # Sort by probability descending
    candidates.sort(key=lambda x: x[2], reverse=True)
 
    # Select top-k non-overlapping, non-duplicate French targets
    selected = []
    used_french = set()
 
    for wo_idx, fr_word, prob in candidates:
        if fr_word in used_french:
            continue
        selected.append((wo_idx, fr_word, prob))
        used_french.add(fr_word)
        if len(selected) >= num_substitutions:
            break
 
    return selected

In [5]:
# =========================
# 7. FULL PIPELINE
# =========================
def process_sentence(wolof_text, french_text, entities, lexicon):
    """
    Full pipeline for one sentence pair:
    1. Protect entities
    2. Substitute Wolof words with French equivalents from lexicon
       (constrained to words present in the paired French sentence)
    3. Informalize remaining Wolof tokens
    4. Restore protected spans
    """
    if not isinstance(wolof_text, str) or not wolof_text.strip():
        return wolof_text
 
    # Step 1: protect entities
    masked, saved = mask_protected(wolof_text, entities)
 
    # Tokenize
    wo_tokens = word_tokens(masked)
    fr_tokens = word_tokens(french_text)
    sentence_len = len(wo_tokens)

    num_substitutions = max(
        1,
        round(sentence_len * 0.2)   # 20% of words
    )
    
    # Step 2: find substitution candidates
    selected = get_substitution_candidates(
        wo_tokens, fr_tokens, lexicon, num_substitutions=num_substitutions
    )
    substitution_map = {wo_idx: fr_word for wo_idx, fr_word, _ in selected}
    substituted_french = set(fr_word for _, fr_word, _ in selected)
 
    # Step 3: rebuild token list with substitutions + informalization
    output_tokens = []
    for i, tok in enumerate(wo_tokens):
        if i in substitution_map:
            output_tokens.append(substitution_map[i])
        elif re.fullmatch(r"__protected_\d+__", tok, flags=re.IGNORECASE):
            output_tokens.append(tok)
        else:
            output_tokens.append(normalize_wolof_token(tok))
 
    result = " ".join(output_tokens)
 
    # Step 4: restore protected spans
    result = restore_protected(result, saved)
    result = re.sub(r"\s+", " ", result).strip()
 
    return result

In [ ]:
from transformers import pipeline as hf_pipeline
import torch

print("Loading dataset...")
parallel_df = pd.read_parquet(TRAIN_PARQUET_PATH)
print(f"Loaded {len(parallel_df)} sentence pairs")

# Reuse the Linux/WSL-built lexicon unless rebuilding is intentional.
import pickle
REBUILD_EFLOMAL_LEXICON = False
if REBUILD_EFLOMAL_LEXICON:
    if eflomal is None:
        raise RuntimeError('Install/run Eflomal under Linux or WSL to rebuild the lexicon.')
    lexicon = build_lexicon(
        parallel_df, wolof_col='wolof', french_col='french',
        min_count=3, min_prob=0.1, batch_size=5000
    )
else:
    with open(LEXICON_PATH, 'rb') as handle:
        lexicon = pickle.load(handle)
    print(f'Reused {LEXICON_PATH} with {len(lexicon):,} Wolof entries')

# Optional NER protection. Keep this disabled unless all benchmark methods
# are regenerated with the same setting.
USE_NER = False
if USE_NER:
    device = 0 if torch.cuda.is_available() else -1
    print("Loading NER model...")
    ner_pipeline = hf_pipeline(
        "token-classification",
        model="davlan/xlm-roberta-large-ner-hrl",
        aggregation_strategy="simple",
        device=device
    )
    def data_generator(df, column):
        for text in df[column]:
            yield str(text) if pd.notna(text) else ""
    print("Extracting entities...")
    all_entities = list(tqdm(
        ner_pipeline(data_generator(parallel_df, "wolof"), batch_size=32),
        total=len(parallel_df)
    ))
else:
    all_entities = [[] for _ in range(len(parallel_df))]

# Generate code-switched output
print("Generating code-switched text...")
outputs = []
for wolof_text, french_text, entities in tqdm(
    zip(parallel_df["wolof"], parallel_df["french"], all_entities),
    total=len(parallel_df)
):
    out = process_sentence(
        wolof_text=wolof_text,
        french_text=french_text,
        entities=entities,
        lexicon=lexicon
    )
    outputs.append(out)

parallel_df["informal code switched"] = outputs

# Diagnostics
n_unchanged = sum(1 for o, w in zip(outputs, parallel_df["wolof"])
                    if word_tokens(str(o)) == word_tokens(str(w)))
print(f"\nSentences with at least one substitution: "
        f"{len(outputs) - n_unchanged}/{len(outputs)} "
        f"({100*(len(outputs)-n_unchanged)/len(outputs):.1f}%)")

print("\nSample outputs:")
print(parallel_df[["french", "wolof", "informal code switched"]].head(10).to_string())

Loading dataset...
Loaded 17777 sentence pairs
Running eflomal alignment on 17777 sentence pairs...


100%|██████████| 4/4 [00:12<00:00,  3.14s/it]

Lexicon built: 1989 Wolof words with French translations
Lexicon saved to: artifacts/lexicon.pkl


In [18]:
parallel_df.iloc[6].to_dict()

{'french': 'Et comme l’on a coutume de le dire dans nos contrées : Si tu ne sais pas où tu vas, retourne d’ où tu viens et c’est bien normal que nous revenons à la cage de départ pour le grands dame des consommateurs.\n',
 'wolof': 'Ni ñu ko tàmm a waxe fii : boo xamatul foo jëm, dellul fa nga jóge woon, baax na lool ñu dellu fa ñu jóge woon ngir bégal way-jëfandiku yi.\n',
 'sources': 'mafand',
 'informal code switched': 'que gnou ko tamm de wakhé fi bo khamatoul fo jeum delloul où nga diogué wone bakh a lol gnou dellou fa gnou diogué wone pour bégal way jeufandikou yi'}

In [19]:
parallel_df.iloc[12026].to_dict()

{'french': 'Une seconde explication est le fait que les femmes seraient plus sensibles à la pression sociale.,\n',
 'wolof': 'Ñaareelu lay wi mooy jigéen ñi la bundxatalug pénc mi di gën a sonal.\n',
 'sources': 'mafand',
 'informal code switched': 'gnarelou lay wi est femmes gni la boundkhataloug pénc mi di plus a sonal'}

In [20]:
parallel_df.iloc[126].to_dict()

{'french': 'Pour nous vendre l’expertise de Fieldstone, le spécialiste de la privatisation des entreprises publiques en Afrique, la Présidence qui recevait son Boss hier en audience en présence de Makhtar Cissé, nous la présente comme: une société conseillère financière de la commission économique des Nations Unies.\n',
 'wolof': 'Ngir jaay ñu xarañte Fildston, ki xam-xamam màcc ci wàllu àntarpiriisu jàmbur ci Afrig, këru nguur gi doon dalal seen njiit démb ci kanamu Maxtaar Siise, di ñu ko wonee ne : genn kuréel guy xelal ci wàllu koppar kuréelu koom gu Nasiyoŋ sini.\n',
 'sources': 'mafand',
 'informal code switched': 'pour vendre gnou kharagnté Fildston le kham khamam mathié thi wallou antarpirisou diambour thi Afrig keurou ngour gui done dalal sene ndjit hier thi kanamou Maxtaar Siise di gnou ko woné né guenne kourel gouy khelal thi wallou koppar kourelou économique de Nasiyong sini'}

## M2 benchmark finalization

This section converts the Eflomal-derived lexicon into a reproducible benchmark dataset. It uses the shared M1 spelling engine, records every alignment substitution, exports the common schema, and creates a manifest and 100-row review sample.

In [ ]:
from src.config import (
    LEXICON_PATH, M2_EFLOMAL_MANIFEST_PATH, M2_EFLOMAL_PATH,
    M2_EFLOMAL_REVIEW_PATH, TRAIN_PARQUET_PATH,
)
from src.generation.alignment_benchmarks import (
    METHOD_M2, generate_m2_dataset, write_alignment_benchmark,
)

BENCHMARK_SEED = 2026
M2_SUBSTITUTION_RATIO = 0.20
REVIEW_SAMPLE_SIZE = 100

entities_by_row = {
    row_id: [(entity["start"], entity["end"]) for entity in entities]
    for row_id, entities in enumerate(all_entities)
}
m2_pairs = generate_m2_dataset(
    parallel_df, lexicon=lexicon, seed=BENCHMARK_SEED,
    substitution_ratio=M2_SUBSTITUTION_RATIO,
    entities_by_row=entities_by_row,
)
assert len(m2_pairs) == len(parallel_df)
assert m2_pairs["source_id"].is_unique
assert m2_pairs["formal_wolof"].tolist() == parallel_df["wolof"].fillna("").astype(str).tolist()
m2_pairs.head()

In [ ]:
m2_diagnostics = {
    "rows": len(m2_pairs),
    "changed_rows": int(m2_pairs["changed"].sum()),
    "changed_rate": float(m2_pairs["changed"].mean()),
    "rows_with_alignment": int(m2_pairs["alignment_edit_count"].gt(0).sum()),
    "alignment_edits": int(m2_pairs["alignment_edit_count"].sum()),
    "spelling_edits": int(m2_pairs["spelling_edit_count"].sum()),
}
m2_diagnostics

In [ ]:
m2_manifest = write_alignment_benchmark(
    m2_pairs,
    output_path=M2_EFLOMAL_PATH,
    review_path=M2_EFLOMAL_REVIEW_PATH,
    manifest_path=M2_EFLOMAL_MANIFEST_PATH,
    method=METHOD_M2,
    input_path=TRAIN_PARQUET_PATH,
    artifacts={"eflomal_lexicon": LEXICON_PATH},
    parameters={
        "benchmark_seed": BENCHMARK_SEED,
        "substitution_ratio": M2_SUBSTITUTION_RATIO,
        "use_ner": USE_NER,
        "candidate_constraint": "French tokens from the paired row",
        "selection": "highest Eflomal probability with lexical tie-break",
        "postprocessing": "shared M1 Beqi-inspired spelling rules",
    },
    review_size=REVIEW_SAMPLE_SIZE,
    review_seed=BENCHMARK_SEED,
)
m2_manifest